In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
FT_2ND_YEAR_SRC = pd.read_csv(r"S:\15.09.26\30919829_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
FT_1ST_YEAR_SRC = pd.read_csv(r"S:\2026\01-Jan-26\01.01.26\28687976_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
BEINDATA_SRC = pd.read_csv(r"S:\15.09.26\15092026_BEINDATANEWRPT.csv", encoding='cp1256',dtype='str')

In [ ]:
first_year_from = '2025-01-01'
first_year_to = '2025-08-31'
second_year_from = '2026-01-01'
second_year_to = '2026-08-31'

In [ ]:
ft_2nd_year = FT_2ND_YEAR_SRC.copy()
ft_1st_year = FT_1ST_YEAR_SRC.copy()
beindata = BEINDATA_SRC.copy()

showrooms = [ "Maadi showroom","Mohandeseen Showroom" ]
dth_types = ['beIN Quartar Installment', 'CNE Subscriber', 'MCE staff (CNE staff)',
                'BeIN sports CC', 'beIN Bi Installment', 'Corporate Subscriber', 'Temp',
                'Bein NC', 'Bulk DTH customer', 'beIN Installment Sub', 'Charge Back']

plan_filter = (
            beindata["PLAN"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            beindata["PLAN"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            beindata["PLAN"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )

invoice_types = ['Subscription Invoice']

2025

In [ ]:
ft_1st_year['Created Date'] = pd.to_datetime(ft_1st_year['Created Date'], dayfirst=True)

ft_1st_year = ft_1st_year.loc[ft_1st_year['Created Date'].between(pd.to_datetime(first_year_from),pd.to_datetime(first_year_to))]
ft_1st_year = ft_1st_year.loc[ft_1st_year['Default Entity Type'].isin(['CNE Dealer'])]
# ft_1st_year = ft_1st_year.loc[ft_1st_year['Subscriber Type'].isin(dth_types)]
ft_1st_year = ft_1st_year.loc[(ft_1st_year['Doc Type'] =='JV') & (ft_1st_year['Doc Status'] =='Posted')]
# ft_1st_year = ft_1st_year.loc[ft_1st_year['Payment Flag']=='Normal payment']



print(f'2025 walk in subs: {ft_1st_year.shape[0]}')


2026

In [ ]:
ft_2nd_year['Created Date'] = pd.to_datetime(ft_2nd_year['Created Date'], dayfirst=True)
ft_2nd_year = ft_2nd_year.loc[ft_2nd_year['Created Date'].between(pd.to_datetime(second_year_from),pd.to_datetime(second_year_to))]
ft_2nd_year = ft_2nd_year.loc[(ft_2nd_year['Doc Type'] =='JV') & (ft_2nd_year['Doc Status'] =='Posted')]
# ft_2nd_year = ft_2nd_year.loc[ft_2nd_year['PAYMENT_FLAG']=='Normal payment']

In [ ]:
beindata = beindata.loc[plan_filter]
beindata = beindata.sort_values(['Customer Number' , 'STATUS'], ascending=[True,True])
beindata = beindata.drop_duplicates(subset=['Customer Number'])
beindata = beindata.dropna(subset=['STATUS'])

In [ ]:
# display(ft2025.columns)
# display(beindata.columns)

In [ ]:
ft_1st_year = ft_1st_year.merge(right=beindata[['Customer Number','PLAN','STATUS']], left_on='Subscriber Nr', right_on='Customer Number', how='inner')
ft_1st_year.shape[0]

In [ ]:
ft_1st_year['is_in_2026'] = False


ft_1st_year = ft_1st_year.merge(right=ft_2nd_year[['Subscriber Nr','Collecting Entity','Created Date','Amount','Default Entity Type']], on='Subscriber Nr',how='left')

ft_1st_year.loc[ft_1st_year['Default Entity Type_y'].isin(['CNE Dealer']),'is_in_2026'] = True


# ft_1st_year['isCne'] = 'beIN'
# ft_1st_year.loc[ft_1st_year['Collecting Entity_y'].str.lower().str.contains('cne',na=False),'isCne']='CNE'
# ft_1st_year.loc[ft_1st_year['Collecting Entity_y'].isin(showrooms),'isCne']='CNE'
# ft_1st_year.loc[ft_1st_year['Collecting Entity_y'].isna(),'isCne']=None


ft_1st_year['Created Date_y'] = pd.to_datetime(ft_1st_year['Created Date_y'], dayfirst=True)

ft_1st_year =  ft_1st_year.sort_values(['Subscriber Nr','Created Date_y'],ascending=[True,False])
ft_1st_year = ft_1st_year.drop_duplicates(subset=['Subscriber Nr'],keep='first')


In [ ]:
ft_1st_year.shape[0]

In [ ]:
ft_1st_year.to_csv("dealers_comparison.csv", index= False)

In [3]:
df = pd.read_csv(r"C:\Users\mturky\Downloads\30934163_BEINDATANEWRPT.CSV")
df

,Customer Number,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance
0,9999654,beIN Quartar Installment,CNE Head office,1011252,29-11-2019,28-11-2021,beIN Sports,DIS,303293316.0,beIN Decoder 1000s,4.291662e+10,beIN Smartcard 1000s,29-11-2021,3M,0 Dr,.01 Cr,.01 Cr
1,9999654,beIN Quartar Installment,CNE Head office,50896,31-10-2018,30-10-2019,beIN Sports,DIS,303293316.0,beIN Decoder 1000s,4.291662e+10,beIN Smartcard 1000s,31-10-2019,3M,0 Dr,.01 Cr,.01 Cr
2,9999654,beIN Quartar Installment,CNE Head office,1929649,20-01-2022,19-01-2023,Premium,DIS,303293316.0,beIN Decoder 1000s,4.291662e+10,beIN Smartcard 1000s,20-01-2023,3M,0 Dr,.01 Cr,.01 Cr
3,9999475,beIN Bi Installment,CNE Head office,1603175,18-08-2021,17-08-2022,Premium Act BTS 2021,DIS,303293326.0,beIN Decoder 1000s,4.291673e+10,beIN Smartcard 1000s,18-08-2022,12M,0 Dr,0 Dr,0 Dr
4,9999475,beIN Bi Installment,CNE Head office,4227822,07-06-2026,06-03-2027,AddON FWC 2026,DIS,303293326.0,beIN Decoder 1000s,4.291673e+10,beIN Smartcard 1000s,07-03-2027,9M,0 Dr,0 Dr,0 Dr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3071169,10000116,CNE Subscriber,Meet Ghamr,1605970,22-08-2021,21-08-2022,Premium,DIS,302785494.0,beIN Decoder 1000s,4.276843e+10,beIN Smartcard 1000s,22-08-2022,3M,0 Dr,0 Dr,0 Dr
3071170,10000116,CNE Subscriber,Meet Ghamr,2080577,22-08-2022,21-08-2023,Premium,DIS,302785494.0,beIN Decoder 1000s,4.276843e+10,beIN Smartcard 1000s,22-08-2023,3M,0 Dr,0 Dr,0 Dr
3071171,10000116,CNE Subscriber,Meet Ghamr,2563708,22-08-2023,21-08-2024,Premium,DIS,302785494.0,beIN Decoder 1000s,4.276843e+10,beIN Smartcard 1000s,22-08-2024,3M,0 Dr,0 Dr,0 Dr
3071172,10000116,CNE Subscriber,Meet Ghamr,51041,21-08-2018,20-08-2019,beIN Sports,DIS,302785494.0,beIN Decoder 1000s,4.276843e+10,beIN Smartcard 1000s,21-08-2019,3M,0 Dr,0 Dr,0 Dr


In [4]:
df['Outstanding Balance'].value_counts()

Outstanding Balance
0 Dr         2285996
.01 Cr        513605
3 Cr           45663
1 Cr           25080
2 Cr           13165
              ...   
11356 Cr           1
12.79 Dr           1
131.13 Cr          1
231.13 Cr          1
30.23 Cr           1
Name: count, Length: 3413, dtype: int64